# GoodWe EV ChargeOps — Chatbot Inteligente
**EV Challenge 2026 | Sprint 2**

Chatbot assistente para gestão de eletropostos em condomínios, utilizando a API do Hugging Face com system prompt contextualizado e memória de conversa.

---
**Integrantes:**
- Bryan Lugli — RM 571350
- Beckman — RM 573442
- Guilherme — RM 573053

## 1. Instalação de dependências

In [45]:
!pip install huggingface_hub -q

In [46]:
import os
from huggingface_hub import InferenceClient

# Carrega a chave via Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    api_key = secrets.get_secret("HF_API_KEY")
except Exception:
    api_key = os.environ.get("HF_API_KEY")

if not api_key:
    raise ValueError("HF_API_KEY não encontrada. Configure nos Kaggle Secrets.")

# Modelo conversacional compatível com a Inference API do Hugging Face
MODEL = "Qwen/Qwen2.5-7B-Instruct"

client = InferenceClient(model=MODEL, token=api_key)
print("API do Hugging Face configurada com sucesso.")
print(f"Modelo: {MODEL}")

API do Hugging Face configurada com sucesso.
Modelo: Qwen/Qwen2.5-7B-Instruct


## 3. System Prompt — Contexto GoodWe EV ChargeOps

In [47]:
SYSTEM_PROMPT = """
Você é o assistente virtual EV ChargeOps, desenvolvido pela GoodWe para o EV Challenge 2026.
Seu papel é auxiliar moradores, síndicos e administradores de condomínios na gestão inteligente
de eletropostos e carregamento de veículos elétricos. Responda sempre em português.

## Contexto do Sistema
O condomínio possui um sistema GoodWe de gestão de carga com as seguintes características:
- 8 eletropostos disponíveis (pontos A1 a A8)
- Potência total disponível: 48 kW (6 kW por ponto)
- Sistema de balanceamento dinâmico de carga entre os pontos ativos
- Cobrança por kWh consumido, proporcional ao uso de cada unidade
- Horário de pico com restrição: 18h às 21h (potência reduzida a 50% por ponto)
- Regras do condomínio: tempo máximo de ocupação do ponto = 4 horas contínuas

## Dados Atuais do Sistema
- Pontos ocupados agora: A2 (Unidade 302), A5 (Unidade 108)
- Consumo do mês atual por unidade:
  - Unidade 101: 42,3 kWh | Unidade 108: 78,1 kWh | Unidade 204: 15,7 kWh
  - Unidade 302: 95,4 kWh | Unidade 415: 31,2 kWh | Unidade 512: 60,8 kWh
- Status da rede: Carga atual = 12 kW de 48 kW disponíveis (25%) — NORMAL
- Tarifa vigente: R$ 0,85 por kWh

## Suas Responsabilidades
1. Responder perguntas sobre consumo de energia, status dos carregadores e regras do condomínio.
2. Informar disponibilidade de pontos e tempo estimado de liberação.
3. Explicar o sistema de cobrança de forma clara e transparente.
4. Alertar sobre sobrecargas ou uso fora das regras estabelecidas.
5. Respeitar a privacidade: nunca revelar o nome do morador, apenas a unidade.
6. Orientar o síndico sobre relatórios e gestão do sistema.

## Restrições
- Responda APENAS sobre temas relacionados a eletropostos, carregamento de EVs,
  consumo de energia e gestão condominial nesse contexto.
- Se perguntado sobre outros assuntos, redirecione educadamente.
- Use linguagem acessível para moradores não técnicos.
- Formate valores monetários em Reais (R$).

## Exemplos de respostas esperadas (few-shot)
Pergunta: "Quanto vou pagar esse mês?"
Resposta: "Para calcular, preciso saber sua unidade. Com a tarifa de R$ 0,85/kWh, posso informar
o valor exato assim que souber qual é a sua."

Pergunta: "Qual o carregador mais rápido?"
Resposta: "Todos os pontos operam a 6 kW. Fora do horário de pico (antes das 18h ou após as 21h)
você tem a potência completa, que é o mais rápido disponível no sistema."
"""

## 4. Motor do Chatbot — Histórico de Mensagens e Chamada à API

In [48]:
class GoodWeChatbot:
    """
    Chatbot EV ChargeOps da GoodWe.
    Usa chat_completion da Inference API do Hugging Face,
    mantendo histórico de mensagens para contexto contínuo.
    """

    def __init__(self, temperature: float = 0.4, max_tokens: int = 600):
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.historico = []  # Memória da conversa

    def enviar_mensagem(self, mensagem_usuario: str) -> str:
        """Envia mensagem e retorna a resposta do assistente."""

        # Adiciona a mensagem do usuário ao histórico
        self.historico.append({"role": "user", "content": mensagem_usuario})

        # Monta o payload: system prompt + histórico completo
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + self.historico

        # Chamada via chat_completion (modo conversacional)
        resposta = client.chat_completion(
            messages=messages,
            max_tokens=self.max_tokens,
            temperature=self.temperature,
        )

        conteudo = resposta.choices[0].message.content.strip()

        # Salva a resposta no histórico
        self.historico.append({"role": "assistant", "content": conteudo})

        return conteudo

    def limpar_historico(self):
        """Reinicia a conversa."""
        self.historico = []
        print("Histórico limpo. Nova conversa iniciada.")

    def exibir_historico(self):
        """Exibe o histórico formatado."""
        if not self.historico:
            print("Histórico vazio.")
            return
        for msg in self.historico:
            role = "Você" if msg["role"] == "user" else "EV ChargeOps"
            print(f"\n[{role}]\n{msg['content']}")
            print("-" * 60)


chatbot = GoodWeChatbot()
print("Chatbot GoodWe EV ChargeOps inicializado e pronto para uso.")

Chatbot GoodWe EV ChargeOps inicializado e pronto para uso.


## 5. Execução dos Casos de Teste — Sprint 1

Os 5 casos de teste definidos na Sprint 1 são executados automaticamente aqui.
Copie os outputs desta célula para preencher o arquivo `docs/testes_resultados.md`.

In [49]:
casos_de_teste = [
    "Quanto cada morador consumiu este mês?",
    "Como funciona a cobrança do carregamento?",
    "Posso carregar meu carro à noite?",
    "O sistema está sobrecarregado?",
    "Quem está usando o carregador agora?",
]

print("=" * 70)
print("  EXECUÇÃO DOS CASOS DE TESTE — GoodWe EV ChargeOps")
print("=" * 70)

for i, pergunta in enumerate(casos_de_teste, start=1):
    chatbot.limpar_historico()
    print(f"\nTESTE {i}: {pergunta}")
    print("-" * 60)
    resposta = chatbot.enviar_mensagem(pergunta)
    print(f"Resposta:\n{resposta}")
    print("=" * 70)

print("\nTodos os 5 casos de teste executados.")

  EXECUÇÃO DOS CASOS DE TESTE — GoodWe EV ChargeOps
Histórico limpo. Nova conversa iniciada.

TESTE 1: Quanto cada morador consumiu este mês?
------------------------------------------------------------
Resposta:
Com base nos dados fornecidos, aqui está o consumo de energia por unidade este mês:

- Unidade 101: 42,3 kWh
- Unidade 108: 78,1 kWh
- Unidade 204: 15,7 kWh
- Unidade 302: 95,4 kWh
- Unidade 415: 31,2 kWh
- Unidade 512: 60,8 kWh

Se precisar de mais detalhes ou tiver alguma outra pergunta sobre o consumo, estou à disposição!
Histórico limpo. Nova conversa iniciada.

TESTE 2: Como funciona a cobrança do carregamento?
------------------------------------------------------------
Resposta:
A cobrança do carregamento é feita de forma simples e transparente. Você será cobrado pelo consumo de energia em kWh (quilowatts-hora) que seu veículo usa para carregar, e o valor é calculado com base na tarifa atual de R$ 0,85 por kWh.

Aqui está um exemplo de como funciona:

1. **Medição do Co

## 6. Interface Interativa — Conversa Livre

Digite sua pergunta e converse livremente com o chatbot.
Comandos: `sair` para encerrar | `limpar` para nova conversa | `histórico` para ver o log.

In [ ]:
chatbot.limpar_historico()

print("\nGoodWe EV ChargeOps — Assistente Virtual")
print("Digite sua dúvida sobre o sistema de eletropostos do condomínio.")
print("=" * 60)

while True:
    entrada = input("\nVocê: ").strip()

    if not entrada:
        continue
    elif entrada.lower() == "sair":
        print("Encerrando o assistente. Até logo!")
        break
    elif entrada.lower() == "limpar":
        chatbot.limpar_historico()
        continue
    elif entrada.lower() == "histórico":
        chatbot.exibir_historico()
        continue

    print("\nEV ChargeOps:")
    resposta = chatbot.enviar_mensagem(entrada)
    print(resposta)

Histórico limpo. Nova conversa iniciada.

GoodWe EV ChargeOps — Assistente Virtual
Digite sua dúvida sobre o sistema de eletropostos do condomínio.


## 7. Demonstração de Memória de Contexto (Multi-turn)

Mostra que o chatbot lembra o contexto ao longo de perguntas encadeadas.

In [ ]:
print("Demonstração: conversa encadeada com memória de contexto")
print("=" * 60)

chatbot.limpar_historico()

perguntas_encadeadas = [
    "Sou da unidade 302. Quanto consumi este mês?",
    "E qual seria o valor que vou pagar?",
    "Tem algum ponto disponível pra eu usar agora antes do horário de pico?",
]

for pergunta in perguntas_encadeadas:
    print(f"\nVocê: {pergunta}")
    resposta = chatbot.enviar_mensagem(pergunta)
    print(f"EV ChargeOps: {resposta}")
    print("-" * 60)

print("\nDemonstração concluída.")
print("O chatbot manteve o contexto da unidade 302 sem que fosse necessário repetir essa informação.")